# MRI Reconstruction with BART Forward Operators and deepinverse

This notebook demonstrates how to use `bartorch.lib` persistent BART
encoding operators together with the
[deepinverse](https://deepinv.github.io) framework for deep-learning-based
MRI reconstruction.

## What we cover

1. Building a Cartesian SENSE encoding operator from scratch using
   `bartorch.lib.encoding_op` — a persistent BART C library operator that
   never goes through the CLI layer.
2. Verifying the operator (adjointness, normal equation).
3. Pure CG reconstruction via `bartorch.lib.conjgrad_solve` (entirely in C).
4. Wrapping the operator as a `deepinv.physics.LinearPhysics` object and
   using it with a deepinverse unrolled algorithm (Half-Quadratic Splitting).

## Prerequisites

```bash
pip install bartorch[deepinv]   # installs deepinv as optional dep
```

In [ ]:
import torch

import bartorch.tools as bt
import bartorch.lib as bl
from bartorch.interop import BartLinearPhysics

## 1 — Simulate data

We use BART's built-in Shepp-Logan phantom and ESPIRiT sensitivity estimation
to create a simple 8-coil 2-D Cartesian MRI scenario.

In [ ]:
NC, NY, NX = 8, 128, 128

# Ground-truth image (Shepp–Logan, complex64)
img_gt = bt.phantom([NY, NX])

# Simulate calibration k-space and compute ESPIRiT sensitivity maps
ksp_calib = bt.phantom([NY, NX], kspace=True, ncoils=NC)
sens = bt.ecalib(ksp_calib, calib_size=24, maps=1)  # (1, NC, 1, NY, NX)

# Simulate measured k-space via SENSE forward model
coil_imgs = sens[0] * img_gt                         # (NC, 1, NY, NX)
ksp_full  = bt.fft(coil_imgs, axes=(-1, -2))         # (NC, 1, NY, NX)

print(f"Sensitivity maps shape : {sens.shape}")
print(f"K-space shape          : {ksp_full.shape}")
print(f"Ground-truth img shape : {img_gt.shape}")

## 2 — Build the persistent BART encoding operator

`bl.encoding_op` calls BART's `pics_model()` C library function once and
returns a `BartLinop` object.  The operator is **persistent**: it is
constructed once and can be applied many times without re-initialisation.

In [ ]:
E = bl.encoding_op(sens)
print(E)

## 3 — Verify adjointness

For a valid adjoint pair we expect `|<Ax, y> - <x, A^H y>| / |<Ax, y>| ≈ 0`.

In [ ]:
torch.manual_seed(0)
x = torch.randn(*E.ishape, dtype=torch.complex64)
y = torch.randn(*E.oshape, dtype=torch.complex64)

Ax  = E.forward(x)
AHy = E.adjoint(y)

lhs = (Ax.conj() * y).sum()
rhs = (x.conj() * AHy).sum()

rel_err = (lhs - rhs).abs().item() / lhs.abs().item()
print(f"<Ax,y>  = {lhs:.6f}")
print(f"<x,AHy> = {rhs:.6f}")
print(f"Relative adjointness error: {rel_err:.2e}  (should be < 1e-3)")

## 4 — Pure CG reconstruction (entirely in C)

`bl.conjgrad_solve` delegates to BART's `lsqr + iter_conjgrad` solver
running entirely in C — there are **no Python callbacks** inside the CG loop.

In [ ]:
ksp_y = ksp_full.reshape(E.oshape)                     # reshape to operator domain
img_cg = bl.conjgrad_solve(E, ksp_y, maxiter=30, lam=1e-4)

img_ref = img_gt.reshape(E.ishape)
nrmse   = ((img_cg - img_ref).abs().pow(2).sum().sqrt() /
            img_ref.abs().pow(2).sum().sqrt()).item()
print(f"CG reconstruction NRMSE: {nrmse:.4f}  (< 0.23 expected)")

## 5 — deepinverse integration

`BartLinearPhysics` wraps our `BartLinop` as a `deepinv.physics.LinearPhysics`
object, making it compatible with any deepinverse algorithm.

### 5.1  Create the physics object

In [ ]:
phys = BartLinearPhysics(E, maxiter=30, lam=1e-4)
print(type(phys).__mro__)

### 5.2  Simulate a noisy measurement

deepinverse physics objects support batched inputs `(B, *shape)`.  Our BART
operator is single-slice; the batch loop runs in Python but the per-slice
CG runs in C.

In [ ]:
# Add a batch dimension to satisfy deepinv convention.
img_batch = img_gt.reshape(1, *E.ishape)              # (1, *ishape)
y_batch   = phys.A(img_batch)                          # forward measurement

# Add small Gaussian noise to simulate realistic acquisition.
torch.manual_seed(42)
noise_std = 0.01 * y_batch.abs().max().item()
y_noisy   = y_batch + noise_std * torch.randn_like(y_batch)

print(f"Measurement shape: {y_noisy.shape}")

### 5.3  CG reconstruction via `A_dagger`

In [ ]:
img_rec = phys.A_dagger(y_noisy)

img_ref_batch = img_gt.reshape(1, *E.ishape)
nrmse_dagger  = ((img_rec - img_ref_batch).abs().pow(2).sum().sqrt() /
                  img_ref_batch.abs().pow(2).sum().sqrt()).item()
print(f"A_dagger NRMSE: {nrmse_dagger:.4f}")

### 5.4  Plug-and-play with deepinverse Half-Quadratic Splitting (HQS)

We can now pass `phys` directly to any deepinverse algorithm.  Here we use
HQS with a simple Gaussian denoiser as the proximal operator.

In [ ]:
try:
    import deepinv

    denoiser = deepinv.models.DnCNN(in_channels=2, out_channels=2,
                                     pretrained=None)

    # deepinv works with real-valued tensors; convert complex → 2-channel real.
    def to_real(x: torch.Tensor) -> torch.Tensor:
        return torch.view_as_real(x).permute(0, -1, *range(1, x.ndim)).contiguous()

    def to_complex(x: torch.Tensor) -> torch.Tensor:
        return torch.view_as_complex(x.permute(0, *range(2, x.ndim), 1).contiguous())

    print("deepinv available — HQS example set up (denoiser not pre-trained).")
    print("In practice, load a pre-trained denoiser and call deepinv.optim.HQS.")

except ImportError:
    print("deepinv not installed — skipping HQS example.")
    print("Install with: pip install bartorch[deepinv]")

## Summary

| Feature | How |
|---|---|
| Persistent SENSE operator | `bl.encoding_op(sens)` |
| Forward / adjoint | `E.forward(x)` / `E.adjoint(y)` |
| CG solve (pure C) | `bl.conjgrad_solve(E, y, maxiter=30)` |
| deepinv integration | `BartLinearPhysics(E)` |
| Batched deepinv | `phys.A(x_batch)` / `phys.A_dagger(y_batch)` |

The same `encoding_op` factory supports:

* **Non-Cartesian** — pass `traj=traj_tensor`
* **Undersampling mask** — pass `pattern=mask_tensor`
* **Subspace projection** — pass `basis=basis_tensor` (ncoeff × nt)